# Python Excel Automation: แยกข้อมูลด้วย xlwings #5

**คำอธิบาย:** เรียนรู้วิธีแยกข้อมูลเป็นหลาย tab ด้วย xlwings

**หัวข้อที่ครอบคลุม:**
1. เปิดไฟล์ข้อมูลดิบ
2. ดึงค่าที่ไม่ซ้ำจากคอลัมน์ที่ระบุ
3. กรองข้อมูลสำหรับแต่ละค่าที่ไม่ซ้ำ
4. สร้าง sheet ใหม่สำหรับแต่ละค่า
5. เขียนข้อมูลที่กรองแล้วลงแต่ละ sheet
6. ปรับความกว้างคอลัมน์อัตโนมัติ
7. นำทาง worksheet

> **หมายเหตุ:** วิดีโอต้นฉบับใช้ `.api.AutoFilter()` และ `.api.SpecialCells()` ซึ่งใช้ได้เฉพาะ Windows (COM)
> Notebook นี้ใช้ **pandas** สำหรับการกรองที่ทำงานได้ทั้งบน macOS และ Windows

In [ ]:
# 📦 ติดตั้ง package ที่จำเป็น (ข้ามถ้าติดตั้งแล้ว)
try:
    import xlwings
    import pandas
    print("✅ ติดตั้ง package แล้ว")
except ImportError:
    %pip install xlwings pandas

✅ Packages already installed.


In [ ]:
# 📚 Import ไลบรารี
# xlwings — เปิด Excel และเขียนข้อมูลลง sheet
# pandas — กรอง/จัดกลุ่มข้อมูล (แทนที่ .api.AutoFilter ที่ใช้ได้เฉพาะ Windows)
import xlwings as xw
import pandas as pd

## 1. เปิด Excel Workbook

In [ ]:
# 📥 ดาวน์โหลดไฟล์ข้อมูลตัวอย่างถ้ายังไม่มี
import os
import requests

file_name = "Random Sales Data.xlsx"
url = "https://raw.githubusercontent.com/ddeshar/python-for-excel-manage/main/Random%20Sales%20Data.xlsx"

if os.path.exists(file_name):
    print(f"✅ '{file_name}' มีอยู่แล้ว ข้ามการดาวน์โหลด")
else:
    print(f"📥 ไม่พบ '{file_name}' กำลังดาวน์โหลด...")
    response = requests.get(url)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)
    print(f"✅ ดาวน์โหลด '{file_name}' สำเร็จ!")

✅ 'Random Sales Data.xlsx' already exists. Skipping download.


In [ ]:
# 📂 เปิด Excel workbook ด้วย xlwings
# xw.Book() เปิดไฟล์และสร้างการเชื่อมต่อกับ Excel
file_name = "Random Sales Data.xlsx"
wkb = xw.Book(file_name)
print(f"เปิดแล้ว: {wkb.name}")
print(f"Sheet ทั้งหมด: {[s.name for s in wkb.sheets]}")

Opened: Random Sales Data.xlsx
Sheets: ['Sheet1', 'Order', 'Filtered']


## 2. อ่านข้อมูลเข้า Pandas

In [ ]:
# 📊 อ่านข้อมูล Excel เป็น pandas DataFrame
# .current_region ตรวจจับขอบเขตข้อมูลอัตโนมัติ (เหมือน Ctrl+Shift+End)
# .options(pd.DataFrame, header=1) แปลงเป็น DataFrame โดยใช้แถวแรกเป็นหัวข้อ
source_sheet = wkb.sheets[0]
df = source_sheet.range('A1').current_region.options(pd.DataFrame, header=1).value
print(f"ขนาดข้อมูล: {df.shape}")
print(f"คอลัมน์: {list(df.columns)}")
df.head()

Data shape: (5684, 8)
Columns: ['Location', 'Region', 'Customer', 'Order Date', 'Product', 'Quantity', 'Price', 'Total Sale Amount']


,Location,Region,Customer,Order Date,Product,Quantity,Price,Total Sale Amount
Sales Representative,,,,,,,,
Sara Snyder,Massachusetts,East,Raymond Young,2016-01-01 00:00:00,"Bravo II Megaboss 12-Amp Hard Body Upright, Re...",6.0,12.42,74.52
Sara Snyder,New York,East,Helen Dean,2016-01-01 00:00:00,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",7.0,12.42,86.94
Diane Gonzalez,Washington,West,Shirley Chavez,2016-01-01 00:00:00,Acme Hot Forged Carbon Steel Scissors with Nic...,2.0,16.32,32.64
Sara Snyder,New Jersey,East,Brian Ryan,2016-01-01 00:00:00,Bretford CR4500 Series Slim Rectangular Table,1.0,12.42,12.42
Sara Snyder,New Jersey,East,Benjamin Willis,2016-01-01 00:00:00,Eldon Fold 'N Roll Cart System,3.0,17.83,53.49


## 3. ดึงค่าที่ไม่ซ้ำจากคอลัมน์

In [ ]:
# 🔑 ดึงค่าที่ไม่ซ้ำจากคอลัมน์ — ค่าเหล่านี้จะเป็นชื่อ sheet
# .unique() คืนค่าที่ไม่ซ้ำทั้งหมดในคอลัมน์
# เราจะสร้าง 1 sheet ต่อ 1 ค่าที่ไม่ซ้ำ
column_name = 'Location'  # ปรับชื่อคอลัมน์ให้ตรงกับข้อมูลของคุณ
unique_values = df[column_name].unique()
print(f"ค่าที่ไม่ซ้ำใน '{column_name}': {len(unique_values)}")
print(unique_values)

Unique values in 'Location': 8
['Massachusetts' 'New York' 'Washington' 'New Jersey' 'Oregon'
 'California' 'Connecticut' 'Nevada']


In [ ]:
# 📊 นับจำนวนค่า — แสดงจำนวนแถวของแต่ละค่าที่ไม่ซ้ำ
# มีประโยชน์ในการเข้าใจการกระจายข้อมูลก่อนแยก
print(df[column_name].value_counts())

Location
New York         1295
Massachusetts    1207
Washington        813
New Jersey        761
California        567
Oregon            564
Connecticut       335
Nevada            142
Name: count, dtype: int64


## 4. แยกข้อมูลเป็นหลาย Sheet
กรองข้อมูลด้วย pandas แล้วเขียนแต่ละกลุ่มลง sheet ใหม่ด้วย xlwings

In [ ]:
# 👀 ดูตัวอย่าง — ดูว่าแต่ละ sheet จะมีกี่แถวก่อนสร้าง
# str(value)[:31] ตัดชื่อให้เหลือ 31 ตัวอักษร (ข้อจำกัดชื่อ sheet ของ Excel)
for value in unique_values:
    count = len(df[df[column_name] == value])
    sheet_name = str(value)[:31]
    print(f"  '{sheet_name}': {count} แถว")

  'Massachusetts': 1207 rows
  'New York': 1295 rows
  'Washington': 813 rows
  'New Jersey': 761 rows
  'Oregon': 564 rows
  'California': 567 rows
  'Connecticut': 335 rows
  'Nevada': 142 rows


In [ ]:
# ✂️ แยกข้อมูล — สร้าง sheet แยกสำหรับแต่ละค่าที่ไม่ซ้ำ
# สำหรับแต่ละค่าที่ไม่ซ้ำ:
#   1. กรอง DataFrame ด้วย pandas
#   2. สร้าง sheet ใหม่ใน workbook
#   3. เขียนข้อมูลที่กรองแล้วลง sheet นั้น
#   4. ปรับความกว้างคอลัมน์อัตโนมัติ

# เชื่อมต่อใหม่ถ้า Excel ถูกปิดระหว่างเซลล์
try:
    _ = wkb.name  # ทดสอบการเชื่อมต่อ
except Exception:
    print("⚠️ การเชื่อมต่อ Excel หายไป กำลังเปิด workbook ใหม่...")
    wkb = xw.Book(file_name)
    source_sheet = wkb.sheets[0]

# ข้าม sheet ที่มีอยู่แล้ว (เช่น ถ้ารันเซลล์นี้ซ้ำ)
existing_sheets = [s.name for s in wkb.sheets]

for value in unique_values:
    # กรองข้อมูลสำหรับค่านี้ด้วย pandas
    filtered_df = df[df[column_name] == value]
    
    # สร้างชื่อ sheet (Excel จำกัด: 31 ตัวอักษร)
    sheet_name = str(value)[:31]
    
    # ข้ามถ้า sheet มีอยู่แล้ว
    if sheet_name in existing_sheets:
        print(f"⏭️  Sheet '{sheet_name}' มีอยู่แล้ว — ข้าม")
        continue
    
    # เพิ่ม sheet ใหม่หลัง sheet สุดท้าย
    new_sheet = wkb.sheets.add(name=sheet_name, after=wkb.sheets[-1])
    
    # เขียน DataFrame ที่กรองแล้วลง sheet ใหม่ (ไม่มีคอลัมน์ index)
    new_sheet.range('A1').options(pd.DataFrame, index=False).value = filtered_df
    
    # ปรับความกว้างคอลัมน์อัตโนมัติเพื่อให้อ่านง่าย
    new_sheet.autofit(axis='columns')
    
    print(f"สร้าง sheet: '{sheet_name}' ({len(filtered_df)} แถว)")

print(f"\n✅ เสร็จแล้ว! สร้าง sheet สำหรับ {len(unique_values)} ค่าที่ไม่ซ้ำ")
print(f"Sheet ทั้งหมด: {[s.name for s in wkb.sheets]}")

⚠️ Excel connection lost. Reopening workbook...


CommandError: Command failed:
		OSERROR: -50
		MESSAGE: Parameter error.
		COMMAND: app(pid=82528).open_workbook(workbook_file_name='/Users/macbookpro/Projects/personal/Dss/report/python_excel/notebooks/Random Sales Data.xlsx', update_links=k.do_not_update_links, read_only=None, format=None, password=None, write_reserved_password=None, ignore_read_only_recommended=None, origin=None, delimiter=None, editable=None, notify=None, converter=None, add_to_mru=None, timeout=-1)

## 5. นำทางกลับไป Sheet ต้นทาง

In [ ]:
# 🧭 นำทางกลับไป sheet ต้นทาง
# .activate() ทำให้ sheet นั้นเป็น sheet ที่ใช้งาน/มองเห็นใน Excel
source_sheet.activate()
print(f"Sheet ที่ใช้งาน: {wkb.sheets.active.name}")

Active sheet: Sheet1


## 6. บันทึก Workbook

In [ ]:
# 💾 บันทึก workbook ด้วยชื่อใหม่
# .save('filename') = บันทึกเป็น (สร้างไฟล์ใหม่พร้อม sheet ที่แยกทั้งหมด)
wkb.save('Split_Data_xlwings.xlsx')
print("✅ บันทึกเป็น 'Split_Data_xlwings.xlsx' แล้ว!")

# ปิด workbook เมื่อต้องการ (uncomment)
# wkb.close()

✅ Saved as 'Split_Data_xlwings.xlsx'!


---

## 🎮 Mini Project: แยกข้อมูลด้วย xlwings

ทดสอบการแยกข้อมูลเป็นหลาย Sheet ด้วย xlwings!

> 💡 **คำตอบ:** ดูไฟล์ `answers/12_answers.ipynb`

---

### โจทย์ที่ 1: แยกข้อมูลตาม Product Category 🟢
1. เปิด `Random Sales Data.xlsx` ด้วย xlwings
2. อ่านข้อมูลเป็น DataFrame
3. หา unique `Product Category`
4. สร้าง Sheet ใหม่สำหรับแต่ละ Category
5. เขียนข้อมูลที่กรอง + autofit
6. บันทึกเป็น `split_by_category_xlwings.xlsx`

In [ ]:
# โจทย์ที่ 1: แยกข้อมูลตาม Product Category
# เขียนโค้ดของคุณที่นี่ 👇



### โจทย์ที่ 2: แยก + สรุปจำนวนแต่ละ Sheet 🟡
1. หลังจากแยกข้อมูลแล้ว ให้สร้าง sheet `"Summary"` ที่ท้ายสุด
2. ใน Summary ใส่: Category, จำนวนแถว, ยอดขายรวม
3. Autofit ทุกคอลัมน์
4. บันทึก

> 💡 Hint: `df.groupby(col).agg({'col': 'sum', 'col': 'count'})`

In [ ]:
# โจทย์ที่ 2: แยก + สรุป
# เขียนโค้ดของคุณที่นี่ 👇

